# Atera human colorectal cancer (fresh frozen) with aligned H&E: Landscape, Yearbook and a Matrix-linked Clustergram

Explore the pre-processed 10x Genomics **Atera WTA** dataset `Atera_v1_WTA_S4B_Colorectal_Cancer_FF` with Celldega:

1. a **wide Landscape** of the whole tissue with its default clusters,
2. a **Yearbook** of individual cells (query by cluster and gene),
3. a **linked Landscape and Clustergram** of the default clusters, drawn from a **Matrix** whose clustering was pre-computed, with an **Enrich** widget for gene-set enrichment of selected genes.

Each Landscape and the Yearbook can show either the fluorescence morphology image or the dataset's **post-run H&E image**, aligned to it with the 10x Genomics alignment matrix. Cells, transcripts and clusters stay in place on both backgrounds, so expression can be compared directly with tissue histology.

The Matrix is a clusters x genes object with **pre-clustered results stored in an AnnData-Zarr** (dendrograms, dot-plot sizes, marker-ranked views, and value attributes). It is loaded here, not computed, so the Clustergram appears right away. The Clustergram is the visualization of that Matrix.

**Data** (all public):

| What | Where |
|---|---|
| DegaFiles (fluorescence and aligned H&E images, cells, transcripts, expression) | GitHub repository [`celldega_atera-wta-ff-human-crc`](https://github.com/broadinstitute/celldega_atera-wta-ff-human-crc) (one file per tile) |
| Matrix with pre-clustered results (AnnData-Zarr) | Hugging Face only: `Atera/Atera_v1_WTA_S4B_Colorectal_Cancer_FF/cgm/default_clusters.zarr` (and a single-file `.zarr.zip`) |
| Original 10x data | [Atera In Situ Gene Expression, Fresh Frozen Human Colorectal Cancer](https://www.10xgenomics.com/datasets/atera-wta-ff-human-crc) (sample `Atera_v1_WTA_S4B_Colorectal_Cancer_FF`, Atera Onboard Analysis 1.1.1) |

Dataset: *Atera In Situ Gene Expression, Fresh Frozen Human Colorectal Cancer*, Atera In Situ Gene Expression Dataset by Atera Onboard Analysis v1.1.1, 10x Genomics, (2026, November 5). The original data are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) (see the [10x citation guidelines](https://www.10xgenomics.com/support/software/cell-ranger/latest/miscellaneous/cr-citations)). Changes made for Celldega: the Atera output bundle was converted to DegaFiles with `dega.pre.main`, the post-run H&E image was resampled onto the morphology image with 10x's alignment CSV using [`dega.pre.add_aligned_he`](../../../python/aligned_he/), and the default-cluster Matrix was computed as described at the end of this notebook. The [Atera pre-processing with H&E](../../brief_notebooks/Atera_Pre-process_with_HE/) notebook shows these steps.

**Requirements:** `pip install "celldega>=0.28.2"`. The Matrix is read with `anndata`/`zarr`, which Celldega already depends on. Scanpy is only needed to re-create the Matrix (see the end of this notebook).

In [1]:
import re
from importlib.metadata import version
from urllib.request import urlopen

import celldega as dega
import numpy as np

# The H&E image selector and Matrix.read_zarr need Celldega 0.28.2 or newer.
installed = tuple(int(x) for x in re.findall(r"\d+", version("celldega"))[:3])
if installed < (0, 28, 2):
    raise RuntimeError(
        f"This notebook needs celldega>=0.28.2 but {version('celldega')} is installed. "
        'Run: pip install -U "celldega>=0.28.2", then restart the kernel.'
    )

SAMPLE = "Atera_v1_WTA_S4B_Colorectal_Cancer_FF"

HF_BASE = "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/Atera"
# DegaFiles for the Landscape and Yearbook: GitHub (one file per tile)
BASE_URL = "https://raw.githubusercontent.com/broadinstitute/celldega_atera-wta-ff-human-crc/main"
# The Matrix (pre-clustered AnnData-Zarr) is hosted on Hugging Face only.
MATRIX_URL = f"{HF_BASE}/{SAMPLE}/cgm/default_clusters"

# Atera centroids are in microns; this matrix maps microns to image pixels.
with urlopen(f"{BASE_URL}/micron_to_image_transform.csv") as response:
    transform = np.loadtxt(response)

## Wide Landscape overview

Explore the full tissue and its default clusters. This full-width Landscape is a separate widget from the linked view below, so selections and resizing in one leave the other independent.

**Switch to H&E:** click **FLR** (fluorescence) next to **SPATIAL IMG** at the top left and choose **H&E**. Choose **FLR** again to return to the DAPI, boundary, RNA and protein channels.

In [2]:
wide_landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera colorectal cancer — overview", width=0, height=800,
    max_tiles_to_view=25,
)
wide_landscape.layout.width = "100%"
wide_landscape

## Look up cells with Yearbook

Yearbook shows a grid of portraits centered on individual cells, using the same default clusters as the Landscape. Query by **cluster**, **gene**, or both: a cluster query samples its cells, and adding a gene ranks them by expression, highest first. This example starts with the largest default cluster (**3**) ranked by **CLDN4**, a strong marker of that cluster. Change the cluster or gene in the Yearbook controls to look up other cells. The Yearbook has the same **FLR / H&E** selector next to **IMG**, so the portraits can also be shown on the H&E image.

In [3]:
yearbook = dega.viz.Yearbook(
    name="atera_yearbook",
    base_url=BASE_URL,
    front_end_query={
        "cluster": {"attr": "cluster", "value": "3"},
        "gene": "CLDN4",
        "max_cells": 60,
    },
    rows=2, cols=6, portrait_size_um=50,
    dataset_name="Atera colorectal cancer — cell yearbook", width=0, height=520,
)
yearbook.layout.width = "100%"
yearbook

## Load the Matrix with pre-clustered results

The default-cluster Matrix was computed once and saved as an **AnnData-Zarr**, so loading it is fast and needs no Scanpy. It holds:

- **values**: log1p CPM of each gene per default cluster (**pseudo-bulk signatures** of *all* cells), z-scored per gene for display;
- **dot sizes**: the fraction of each cluster's cells expressing each gene;
- **marker-ranked views**: the DIM slider shows the top 1, 2, 3, 5, 10, 25, 50, 75 or 100 markers per cluster (ranked with `scanpy.tl.rank_genes_groups`, Wilcoxon, on all cells); all genes remain available;
- **value attributes**: each cluster column has `n_cells`, and each gene row has `total_counts` and `mean_counts` over all cells.

`Matrix.read_zarr` restores the clustering exactly as saved; nothing is re-clustered.

In [4]:
import tempfile
from pathlib import Path

import zarr

# Download the zipped AnnData-Zarr once (a few MB, kept in the system temp folder) and read it.
MATRIX_NAME = "default_clusters"
local = Path(tempfile.gettempdir()) / "celldega_atera" / SAMPLE / f"{MATRIX_NAME}.zarr.zip"
if not local.is_file():
    local.parent.mkdir(parents=True, exist_ok=True)
    with urlopen(f"{MATRIX_URL}.zarr.zip") as response:
        local.write_bytes(response.read())

with zarr.storage.ZipStore(str(local), mode="r") as store:
    matrix = dega.clust.Matrix.read_zarr(store)

print(f"{matrix.to_df().shape[1]} clusters x {matrix.to_df().shape[0]:,} genes")

# Alternative: read the directory-layout AnnData-Zarr straight from the URL, with no download
# (needs `pip install fsspec aiohttp`; this is also the store a browser reads):
# matrix = dega.clust.Matrix.read_zarr(
#     zarr.storage.FsspecStore.from_url(f"{MATRIX_URL}.zarr", read_only=True)
# )
matrix.meta_col[["n_cells"]].head()

24 clusters x 24,677 genes


,n_cells
3,63820
15,29299
21,27829
10,27295
19,25814


## Linked Landscape and Clustergram

Pass the pre-clustered Matrix to `Clustergram` and link it to a Landscape. Click a **gene row** to show its spatial expression, or a **cluster column** to select that cluster's cells. The **DIM** slider changes how many top markers per cluster are shown; the **dot plot** encodes mean expression by color and fraction of cells expressing by size. This Landscape has its own **FLR / H&E** selector, independent of the overview above.

`enrich=True` adds an **Enrich** widget beside the Clustergram: select genes (for example a branch of the row dendrogram, or genes picked in the Landscape) to run gene-set enrichment, and use its controls to choose the gene-set library, such as CellMarker for cell-type annotation.

In [5]:
clustergram = dega.viz.Clustergram(
    matrix=matrix, width=450, height=500, rank_dim=3,
    viz_mode="dotplot", manual_row_cat=False, manual_col_cat=False,
)
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera colorectal cancer — default clusters", height=600, max_tiles_to_view=25,
)
widgets = dega.viz.spatial_clustergram(landscape, clustergram, enrich=True)
widgets

## How the Matrix was made

The Matrix was pre-computed from the Atera `cell_feature_matrix` and the dataset's default Leiden clustering with standard scverse tools. The steps, in outline:

```python
import anndata as ad, scanpy as sc
from celldega.pre.atera import read_atera_cbg, read_atera_clusters

cells = ad.AnnData(...)            # raw counts + default cluster in cells.obs["cluster"]
# keep clusters with at least 100 cells

# pseudo-bulk signatures from ALL cells (no subsampling)
signatures = sc.get.aggregate(cells, by="cluster", func=["sum", "count_nonzero"])
signatures.obs["n_cells"] = cells.obs["cluster"].value_counts().reindex(signatures.obs_names)
signatures.X = signatures.layers["sum"].astype("float32")
signatures.layers["fraction_expressing"] = (
    signatures.layers["count_nonzero"] / signatures.obs["n_cells"].to_numpy()[:, None]
)
sc.pp.normalize_total(signatures, target_sum=1e6)
sc.pp.log1p(signatures)
signatures.var["total_counts"] = ...   # sum of raw counts per gene over all cells
signatures.var["mean_counts"] = ...    # mean raw counts per gene per cell

# marker ranking on all cells, as in the Scanpy clustering tutorial
ranked = ad.AnnData(cells.X.copy(), obs=cells.obs[["cluster"]].copy(), var=cells.var.copy())
sc.pp.normalize_total(ranked)
sc.pp.log1p(ranked)
sc.tl.rank_genes_groups(ranked, groupby="cluster", method="wilcoxon", n_genes=ranked.n_vars)
signatures.uns["rank_genes_groups"] = ranked.uns["rank_genes_groups"]

matrix = dega.clust.Matrix(
    signatures, col_attr=["cluster", "n_cells"], row_attr=["total_counts", "mean_counts"],
    name="default_clusters",
)
matrix.set_size_matrix(ad.AnnData(signatures.layers["fraction_expressing"],
                                  obs=signatures.obs.copy(), var=signatures.var.copy()))
matrix.norm("row", by="zscore")
matrix.cluster(view="rank_genes_groups", levels=[1, 2, 3, 5, 10, 25, 50, 75, 100])

matrix.write_zarr("default_clusters.zarr")                 # AnnData-Zarr, read with Matrix.read_zarr
matrix.write_dega_files("path/to/DegaFiles", name="default_clusters")  # parquet for the browser
```

Cluster colors are taken from the DegaFiles so the Clustergram and Landscape agree.